In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

# ── Input ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/cinestreamdata.csv')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def parse_dollar(value):
    cleaned = re.sub(r'[$,]', '', value.strip())
    try:
        return int(cleaned)
    except ValueError:
        return None

def scrape_domestic_opening(tconst):
    url = f"https://www.boxofficemojo.com/title/{tconst}/"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  ERROR fetching {url}: {e}")
        return None

    soup = BeautifulSoup(res.text, 'html.parser')

    # Find the span with text "Domestic Opening" then get the money span next to it
    for span in soup.find_all('span'):
        if span.get_text(strip=True) == 'Domestic Opening':
            money = span.find_next('span', class_='money')
            if money:
                return parse_dollar(money.get_text(strip=True))

    print(f"  Domestic Opening not found for {tconst}")
    return None

# ── Loop ──────────────────────────────────────────────────────────────────────
results = []
total = len(df)

for i, row in df.iterrows():
    tconst = row['tconst']
    title  = row['title']
    print(f"[{i+1}/{total}] {title} ({tconst})")

    opening = scrape_domestic_opening(tconst)

    results.append({
        'tconst':           tconst,
        'title':            title,
        'domestic_opening': opening
    })
    time.sleep(0.2)

# ── Output ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(results)
out.to_csv('../data/part2_boxofficemojo.csv', index=False)
print(f"\nDone! {len(out)} rows saved to ../data/part2_boxofficemojo.csv")

C:\Users\stefv\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[1/345] Motherless Brooklyn (tt0385887)
[2/345] Alita Battle Angel (tt0437086)
[3/345] Shazam! (tt0448115)
[4/345] Silence (tt0490215)
[5/345] Suburbicon (tt0491175)
[6/345] Chips (tt0493405)
[7/345] Rings (tt0498381)
[8/345] The Last Full Measure (tt0783640)
[9/345] Pet Sematary (tt0837563)
[10/345] Super Troopers 2 (tt0859635)
[11/345] Jungle Cruise (tt0870154)
[12/345] Fantasy Island (tt0983946)
[13/345] The Little Things (tt10016180)
[14/345] Unhinged (tt10059518)
[15/345] Gemini Man (tt1025100)
[16/345] Lightyear (tt10298810)
[17/345] The Invisible Man (tt1051906)
[18/345] Thor Love And Thunder (tt10648342)
[19/345] The Killing Of Two Lovers (tt10702148)
[20/345] Winchester (tt1072748)
[21/345] Bill & Ted Face The Music (tt1086064)
[22/345] Freaky (tt10919380)
[23/345] Old (tt10954652)
[24/345] Wrath Of Man (tt11083552)
[25/345] Spirit Untamed (tt11084896)
[26/345] House Of Gucci (tt11214590)
[27/345] Licorice Pizza (tt11271038)
[28/345] Together Together (tt11285280)
[29/345] Ram